# Experiment 2 — Exploratory Data Analysis

**Aim.** Map the shape of every modality before any model is trained: where does risk concentrate
in the harmonized table, what do the DASS severity bands look like, how do the text corpora and
the face images distribute — and which of those views are honest enough to publish.

**Why this matters.** EDA decides what the models may learn. Class imbalance tells us which
metrics matter (macro-F1, not accuracy); structural missingness in the harmonized table (students
have no workplace-support values, professionals have no sleep/finance values) tells us which
features are population-specific; and cross-corpus overlap discovered here changes how Exp 7
trains its text model.

**Method.** Three processed tracks are loaded side by side — `risk_harmonized` (students +
professionals), `dass42` (questionnaire track) and the three text/image corpora — with shared
helpers from `mindsense.eda` so the notebook stays narrative. Figures are saved to
`reports/figures/exp02_*.png`; two interactive Plotly views are saved for the app's
**Population Insights** page; headline statistics land in `reports/metrics/exp02_eda.json`.
Every figure below carries a 1–2 line takeaway.

In [1]:
# Setup: load every processed table and show the real/synthetic banner (Rule 2).
import json
from collections import Counter
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import seaborn as sns

from mindsense import GLOBAL_SEED
from mindsense.data.dass import BAND_ORDER, SUBSCALE_ITEMS
from mindsense.eda import (
    AGE_BANDS,
    add_age_band,
    association_matrix,
    emotion_trends,
    mutual_information_ranking,
    top_ngrams,
    vader_compound,
    save_plotly_html,
)
from mindsense.nlp.preprocess import word_count
from mindsense.utils.io import repo_path, save_json
from mindsense.utils.plotting import PALETTE, save_fig, style_axes

np.random.seed(GLOBAL_SEED)
sns.set_theme(style="whitegrid", context="notebook")

P = repo_path("data", "processed")
risk = pd.read_parquet(P / "risk_harmonized.parquet")
osmi = pd.read_parquet(P / "osmi_2014.parquet")
dass = pd.read_parquet(P / "dass42.parquet")
sent = pd.read_parquet(P / "sentiment_mh.parquet")
drug = pd.read_parquet(P / "drug_reviews.parquet")
fer = pd.read_parquet(P / "fer2013_manifest.parquet")

counts: Counter[str] = Counter()
for path in sorted(P.glob("*.parquet")):
    if "data_source" in pq.ParquetFile(path).schema.names:
        counts.update(pd.read_parquet(path, columns=["data_source"])["data_source"].value_counts().to_dict())
synthetic = sum(v for k, v in counts.items() if k != "real")
banner = "SYNTHETIC ROWS PRESENT -- never present these numbers as real-world results" if synthetic \
    else f"All {sum(counts.values()):,} processed rows are real downloads"
print(banner)
for name, df in [("risk_harmonized", risk), ("osmi_2014", osmi), ("dass42", dass),
                 ("sentiment_mh", sent), ("drug_reviews", drug), ("fer2013", fer)]:
    print(f"{name:18s} {len(df):>7,} rows")

All 216,247 processed rows are real downloads
risk_harmonized     29,152 rows
osmi_2014            1,251 rows
dass42              38,337 rows
sentiment_mh        50,843 rows
drug_reviews        26,832 rows
fer2013             35,887 rows


## 1 · Tabular risk factors (`risk_harmonized`, N = 29 152)

The harmonized table pairs 27 901 students with 1 251 working professionals under one schema.
**`mh_risk` is a proxy target** (student self-report vs professional treatment indicator), so
every rate below describes this dataset, not clinical prevalence.

In [2]:
# F01 target balance | F02 risk rate by age band x gender
bal = risk.groupby(["population", "mh_risk"]).size().rename("n").reset_index()
fig, ax = plt.subplots(figsize=(9, 5.2))
sns.barplot(data=bal, x="population", y="n", hue="mh_risk", palette=PALETTE[:2], ax=ax)
style_axes(ax, f"Target balance -- mh_risk = 1 in {risk['mh_risk'].mean():.1%} of rows",
           "Population", "Rows")
ax.legend(title="mh_risk", frameon=False, labels=["0 (low risk)", "1 (elevated)"])
save_fig(fig, "exp02_01_target_balance")

r = add_age_band(risk)
age_order = [b for _, _, b in AGE_BANDS if b in set(r["age_band"])]
g = r.groupby(["age_band", "gender"], observed=True)["mh_risk"].mean().reset_index()
fig, ax = plt.subplots(figsize=(10, 5.2))
sns.barplot(data=g, x="age_band", y="mh_risk", hue="gender", order=age_order,
            palette=sns.color_palette(PALETTE, g["gender"].nunique()), ax=ax)
style_axes(ax, "Risk rate by age band and gender", "Age band", "Risk rate (mh_risk = 1)")
ax.set_ylim(0, 1)
ax.legend(title="gender", frameon=False)
save_fig(fig, "exp02_02_age_band")

print("overall rate:", round(risk["mh_risk"].mean(), 4))
print(r.groupby("age_band")["mh_risk"].agg(["mean", "size"]).round(3).to_string())

overall rate: 0.5821
           mean   size
age_band              
14-24     0.681  12329
25-34     0.509  16386
35-44     0.520    358
45-59     0.548     73
60+       0.500      6


**Takeaway (F01):** the positive class is the *majority* (58.2% overall: 58.6% of students,
50.5% of professionals) — accuracy alone would be a misleading headline metric for Exp 6.
**Takeaway (F02):** the 14–24 band (n = 12.3 k) carries the highest rate (~0.68) vs ~0.51 for
25–34; the 45+ bands hold fewer than 400 rows combined and are read as noise, not trend.

In [3]:
# F03 risk rate by gender | F04 risk rate by sleep hours (students only)
g = risk.groupby("gender")["mh_risk"].agg(["mean", "size"]).reset_index()
fig, ax = plt.subplots(figsize=(8, 5.2))
sns.barplot(data=g, x="gender", y="mean", color=PALETTE[0], ax=ax)
ax.bar_label(ax.containers[0], labels=[f"{m:.2f}\n(n={n:,})" for m, n in zip(g["mean"], g["size"])], fontsize=9)
style_axes(ax, "Risk rate by gender", "Gender", "Risk rate")
ax.set_ylim(0, 1.15)
save_fig(fig, "exp02_03_gender")

sl = risk.dropna(subset=["sleep_hours"])
g = sl.groupby("sleep_hours")["mh_risk"].mean().reset_index().sort_values("sleep_hours")
fig, ax = plt.subplots(figsize=(9, 5.2))
ax.plot(g["sleep_hours"], g["mh_risk"], marker="o", linewidth=2, color=PALETTE[0])
for _, row in g.iterrows():
    ax.annotate(f"{row['mh_risk']:.2f}", (row["sleep_hours"], row["mh_risk"]),
                textcoords="offset points", xytext=(0, 8), ha="center", fontsize=9)
style_axes(ax, f"Risk rate by sleep hours (n={len(sl):,}; professionals are structurally NaN)",
           "Sleep hours", "Risk rate")
ax.set_xticks(sorted(sl["sleep_hours"].unique()))
ax.set_ylim(0, 1)
save_fig(fig, "exp02_04_sleep")
print(g.to_string(index=False))

 sleep_hours  mh_risk
         4.0 0.645126
         5.5 0.568818
         7.5 0.595018
         9.0 0.509265


**Takeaway (F03):** female (0.587) and male (0.578) rates are nearly identical; `other` shows
0.667 but has only 21 rows — far too few for any gender-gap claim.
**Takeaway (F04):** 4-hour sleepers show the highest rate (0.65) and 9-hour the lowest (0.51),
but the middle buckets are non-monotonic — sleep is a noisy, students-only feature (professionals
are structurally NaN), not a standalone predictor.

In [4]:
# F05 risk rate by stress/pressure score | F06 risk rate by financial stress
g = risk.dropna(subset=["stress_pressure_score"]).groupby("stress_pressure_score")["mh_risk"].mean().reset_index()
fig, ax = plt.subplots(figsize=(9, 5.2))
ax.plot(g["stress_pressure_score"], g["mh_risk"], marker="o", linewidth=2, color=PALETTE[2])
style_axes(ax, "Risk rate vs stress / pressure score (0-5)", "Stress-pressure score", "Risk rate")
ax.set_ylim(0, 1)
save_fig(fig, "exp02_05_pressure")

fin = risk.dropna(subset=["financial_stress"])
g = fin.groupby("financial_stress")["mh_risk"].mean().reset_index()
fig, ax = plt.subplots(figsize=(9, 5.2))
sns.barplot(data=g, x="financial_stress", y="mh_risk", color=PALETTE[3], ax=ax)
ax.bar_label(ax.containers[0], fmt="%.2f", fontsize=9)
style_axes(ax, f"Risk rate by financial stress (students mainly; n={len(fin):,})",
           "Financial stress (1-5)", "Risk rate")
ax.set_ylim(0, 1)
save_fig(fig, "exp02_06_financial")
print(g.round(3).to_string(index=False))

 financial_stress  mh_risk
              1.0    0.319
              2.0    0.430
              3.0    0.589
              4.0    0.691
              5.0    0.813


**Takeaway (F05):** fractional scores (≈3.3) come from the professional support proxy while
students report integer pressure ratings — one more reason Exp 6 keeps `population` as a feature.
**Takeaway (F06):** the cleanest gradient in the whole EDA — risk rises monotonically from 0.32
(financial stress 1) to 0.81 (stress 5). Financial stress is a first-class feature for Exp 6.

In [5]:
# F07 risk rate by family history | F08 treatment rate by work interference (OSMI 2014)
g = risk.groupby("family_history")["mh_risk"].agg(["mean", "size"]).reset_index()
fig, ax = plt.subplots(figsize=(8, 5.2))
sns.barplot(data=g, x="family_history", y="mean", color=PALETTE[4], ax=ax)
ax.bar_label(ax.containers[0], labels=[f"{m:.2f}\n(n={n:,})" for m, n in zip(g["mean"], g["size"])], fontsize=9)
style_axes(ax, "Risk rate by family history of mental illness", "Family history (0 = no, 1 = yes)", "Risk rate")
ax.set_ylim(0, 1.2)
save_fig(fig, "exp02_07_family")

wi_order = [c for c in ["Never", "Rarely", "Sometimes", "Often", "Not applicable"]
            if c in set(osmi["work_interfere"].dropna())]
g = (osmi[osmi["work_interfere"].isin(wi_order)].groupby("work_interfere")["treatment"]
     .mean().reindex(wi_order).reset_index())
fig, ax = plt.subplots(figsize=(9, 5.2))
sns.barplot(data=g, x="work_interfere", y="treatment", order=wi_order, color=PALETTE[5], ax=ax)
ax.bar_label(ax.containers[0], fmt="%.2f", fontsize=9)
style_axes(ax, "Treatment rate by reported work interference (professionals, n=1,251)",
           "Work interference", "Treatment rate")
ax.set_ylim(0, 1)
save_fig(fig, "exp02_08_work_interfere")
print(g.round(3).to_string(index=False))

work_interfere  treatment
         Never      0.142
        Rarely      0.705
     Sometimes      0.769
         Often       0.85
Not applicable      0.015


**Takeaway (F07):** reported family history lifts the risk rate from 0.550 to 0.617 — a real but
modest association (self-reported; inherited risk is not destiny).
**Takeaway (F08):** treatment rate climbs steeply with interference (Never 0.14 → Rarely 0.71 →
Sometimes 0.77 → Often 0.85); the *Not applicable* bucket sits at 0.02 — exactly the kind of
associational pattern the what-if simulator will show with an explicit "not causal" warning.

In [6]:
# F09 mixed association matrix: Pearson (num x num), Cramer's V (cat x cat), eta (mixed)
num_cols = ["age", "sleep_hours", "stress_pressure_score", "satisfaction_score",
            "work_study_hours", "financial_stress"]
cat_cols = ["gender", "population", "family_history", "diet_quality"]
mat = association_matrix(risk, num_cols, cat_cols)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(mat, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1,
            linewidths=0.4, ax=ax, cbar_kws={"label": "|association| (0-1)"})
ax.set_title("Mixed association matrix -- Pearson / Cramer's V / correlation ratio", fontweight="bold")
save_fig(fig, "exp02_09_association")
print(mat.round(2).to_string())

                        age  sleep_hours  stress_pressure_score  satisfaction_score  work_study_hours  financial_stress  gender  population  family_history  diet_quality
age                    1.00         0.00                   0.09                0.01              0.03              0.10    0.05        0.24            0.01          0.06
sleep_hours            0.00         1.00                   0.04                0.01              0.03              0.00    0.00        0.00            0.01          0.01
stress_pressure_score  0.09         0.04                   1.00                0.11              0.10              0.15    0.03        0.08            0.04          0.09
satisfaction_score     0.01         0.01                   0.11                1.00              0.04              0.07    0.02        0.00            0.00          0.03
work_study_hours       0.03         0.03                   0.10                0.04              1.00              0.08    0.01        0.00           

**Takeaway (F09):** no feature pair exceeds |0.24| (strongest: age × population), so Exp 6 can
keep the full feature set without collinearity concerns; the visible blocks come from structural
NaN patterns (sleep vs workplace-support features separated by `population`).

In [7]:
# F10 mutual-information ranking vs mh_risk
features = ["age", "gender", "population", "sleep_hours", "stress_pressure_score",
            "satisfaction_score", "work_study_hours", "financial_stress", "family_history",
            "support_available", "diet_quality", "suicidal_thoughts"]
categorical = ["gender", "population", "family_history", "diet_quality", "suicidal_thoughts"]
mi = mutual_information_ranking(risk, "mh_risk", features, categorical=categorical, seed=GLOBAL_SEED)
fig, ax = plt.subplots(figsize=(9, 5.2))
mi_sorted = mi.sort_values()
ax.barh(mi_sorted.index, mi_sorted.values, color=PALETTE[0])
style_axes(ax, "Mutual information of features with mh_risk (nats)", "Mutual information")
save_fig(fig, "exp02_10_mi")
print(mi.round(4).to_string())

suicidal_thoughts        0.1407
stress_pressure_score    0.1258
financial_stress         0.0664
age                      0.0315
work_study_hours         0.0255
diet_quality             0.0180
satisfaction_score       0.0123
sleep_hours              0.0045
support_available        0.0026
family_history           0.0023
population               0.0005
gender                   0.0000


**Takeaway (F10):** the leaders are `suicidal_thoughts` (MI 0.141 — kept as an **ablation-only**
feature, never shown as a model input; DECISIONS #9), `stress_pressure_score` (0.126) and
`financial_stress` (0.066). Sleep (0.005) and gender (~0.000) carry little information at this
granularity — coarse bucketing and near-parity, respectively.

In [8]:
# F11 students vs professionals: two defining distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(data=risk, x="age", hue="population", kde=True, bins=30,
             palette=PALETTE[:2], ax=axes[0])
style_axes(axes[0], "Age distribution by population", "Age", "Rows")
sns.histplot(data=risk, x="stress_pressure_score", hue="population", bins=20,
             palette=PALETTE[:2], ax=axes[1])
style_axes(axes[1], "Stress-pressure score by population", "Score", "Rows")
save_fig(fig, "exp02_11_populations")
print(risk.groupby("population")["age"].describe().round(1).to_string())

                count  mean  std   min   25%   50%   75%   max
population                                                    
professional   1251.0  32.1  7.3  18.0  27.0  31.0  36.0  72.0
student       27901.0  25.8  4.9  18.0  21.0  25.0  30.0  59.0


**Takeaway (F11):** professionals are older on average (32.1 ± 7.3 vs 25.8 ± 4.9) but the age
ranges overlap widely, and the 22:1 size ratio means pooled metrics would be student-dominated —
Exp 6 must report per-population metrics alongside the pooled score.

## 2 · DASS-42 questionnaire track (N = 38 337 after validity screening)

Severity bands follow the official cut-offs on **raw 0–42 sums** (no ×2 — DECISIONS #6). The sample
is a self-selected online pool, so shares are *not* population prevalence.

In [9]:
# F12 subscale score distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), sharey=True)
for ax, sub in zip(axes, ["depression", "anxiety", "stress"]):
    scores = dass[f"{sub}_score"]
    sns.histplot(scores, bins=43, kde=True, ax=ax, color=PALETTE[0], edgecolor="white")
    ax.axvline(scores.mean(), color=PALETTE[3], linestyle="--", linewidth=1.5,
               label=f"mean = {scores.mean():.1f}")
    style_axes(ax, f"{sub.capitalize()} subscale (0-42)", "Raw score", "Respondents")
    ax.legend(frameon=False)
save_fig(fig, "exp02_12_dass_dist")
print(dass[["depression_score", "anxiety_score", "stress_score"]].describe().round(1).loc[["mean", "50%"]].to_string())

      depression_score  anxiety_score  stress_score
mean              21.0           15.9          21.1
50%               21.0           15.0          21.0


**Takeaway (F12):** all three subscales skew high relative to clinical norms — consistent with the
self-selection of an online mental-health survey pool (see validity screen in Exp 1).

In [10]:
# F13 severity-band shares per subscale (100% stacked)
shares = pd.DataFrame({
    sub: dass[f"{sub}_band"].value_counts(normalize=True).reindex(BAND_ORDER).fillna(0)
    for sub in ("depression", "anxiety", "stress")
}).T
fig, ax = plt.subplots(figsize=(10, 4.4))
shares.plot(kind="barh", stacked=True, color=PALETTE[:5], ax=ax, width=0.6)
ax.set_xlim(0, 1)
ax.set_xlabel("Share of respondents")
ax.set_title("DASS severity-band shares by subscale", fontsize=13, fontweight="bold")
ax.legend(title="Band", bbox_to_anchor=(1.01, 1), loc="upper left", frameon=False)
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
save_fig(fig, "exp02_13_dass_bands")
print((shares * 100).round(1).to_string())

            Normal  Mild  Moderate  Severe  Extremely severe
depression    22.2   9.7      18.0    16.4              33.8
anxiety       24.4   7.1      18.0    15.5              35.1
stress        29.6  12.6      22.2    21.7              14.0


**Takeaway (F13):** one third of respondents land in *Extremely severe* on depression and anxiety
— the screening skew that Exp 4's ordinal model must handle (and the reason macro-F1 and QWK,
not accuracy, are the reported metrics).

In [11]:
# F14 co-occurrence: depression x anxiety and depression x stress bands
short = {"Normal": "N", "Mild": "Mild", "Moderate": "Mod", "Severe": "Sev", "Extremely severe": "E.Sev"}
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
for ax, other in zip(axes, ["anxiety_band", "stress_band"]):
    ct = pd.crosstab(dass["depression_band"], dass[other]).reindex(
        index=BAND_ORDER, columns=BAND_ORDER, fill_value=0)
    sns.heatmap(ct, annot=True, fmt="d", cmap="Oranges", ax=ax,
                xticklabels=[short[b] for b in BAND_ORDER],
                yticklabels=[short[b] for b in BAND_ORDER])
    ax.set_title(f"Depression x {other.replace('_band', '')}", fontweight="bold")
    ax.set_xlabel(other.replace("_band", "").capitalize() + " band")
    ax.set_ylabel("Depression band")
save_fig(fig, "exp02_14_dass_cooccur")
ct = pd.crosstab(dass["depression_band"], dass["anxiety_band"]).reindex(index=BAND_ORDER, columns=BAND_ORDER, fill_value=0)
diag = np.trace(ct.to_numpy()) / ct.to_numpy().sum()
print(f"band agreement depression x anxiety: {diag:.1%}")

band agreement depression x anxiety: 46.2%


**Takeaway (F14):** exact-band agreement between depression and anxiety is 46.2% — well above
chance but far from identical, so the subscales are correlated yet non-redundant: Exp 4's design
(predict the depression band *from* anxiety/stress items) uses exactly that partial independence.

In [12]:
# F15 depression-band shares by gender and age band
d = add_age_band(dass)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, col in zip(axes, ["gender", "age_band"]):
    ct = (pd.crosstab(d[col], d["depression_band"], normalize="index")
          .reindex(columns=BAND_ORDER, fill_value=0))
    ct.plot(kind="barh", stacked=True, color=PALETTE[:5], ax=ax, width=0.7)
    ax.set_xlim(0, 1)
    ax.set_xlabel("Share within group")
    ax.set_title(f"Depression band by {col}", fontweight="bold")
    ax.legend(title="Band", bbox_to_anchor=(1.01, 1), loc="upper left", frameon=False)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
save_fig(fig, "exp02_15_dass_subgroups")
print((pd.crosstab(d["gender"], d["depression_band"], normalize="index") * 100).round(1).to_string())

depression_band  Extremely severe  Mild  Moderate  Normal  Severe
gender                                                           
female                       33.9   9.7      18.2    21.6    16.5
male                         32.1   9.6      17.4    24.8    16.1
other                        51.2   5.9      16.3     9.7    17.0


**Takeaway (F15):** female and male profiles are nearly identical (Extremely-severe shares
33.9% vs 32.1%); `gender = other` shows 51.2% Extremely-severe but is one of the small groups
Exp 8 must flag as low-support rather than as a discovered disparity.

In [13]:
# F16 country (top 10) severe-rate + urban/rural band profile
top = dass["country"].value_counts().head(10).index
sub = dass[dass["country"].isin(top)].copy()
sub["severe"] = sub["depression_band"].isin(["Severe", "Extremely severe"])
rate = sub.groupby("country")["severe"].mean().sort_values()
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].barh(rate.index, rate.values, color=PALETTE[3])
axes[0].set_xlim(0, 1)
for i, v in enumerate(rate.values):
    axes[0].text(v + 0.01, i, f"{v:.0%}", va="center", fontsize=9)
style_axes(axes[0], "Share in Severe/Extremely-severe depression (top 10 countries)",
           "Share", "Country")
ct = (pd.crosstab(d["urban"], d["depression_band"], normalize="index")
      .reindex(columns=BAND_ORDER, fill_value=0))
ct.plot(kind="barh", stacked=True, color=PALETTE[:5], ax=axes[1], width=0.7)
axes[1].set_xlim(0, 1)
axes[1].set_xlabel("Share within group")
style_axes(axes[1], "Depression band by urban level (1=large metro, 3=rural)", "Urban level", "")
axes[1].legend(title="Band", bbox_to_anchor=(1.01, 1), loc="upper left", frameon=False)
save_fig(fig, "exp02_16_dass_country")
print((rate * 100).round(1).to_string())

country
MY      46.9
IN      49.4
ID      49.9
US      53.2
NONE    53.6
CA      55.6
AU      56.0
GB      56.3
PH      58.1
DE      59.9


**Takeaway (F16):** the severe/Extremely-severe share across the top-10 countries spans a fairly
narrow band (46.9% MY → 59.9% DE, with 53.6% for the missing-country bucket) — differences are
reported as descriptive only; this is a convenience sample, not a national comparison.

In [14]:
# F17 item-level mean scores (42 items, coloured by subscale)
items = [f"Q{i}A" for i in range(1, 43)]
item_sub = {i: s for s, idx in SUBSCALE_ITEMS.items() for i in idx}
colors = [PALETTE[{"depression": 0, "anxiety": 1, "stress": 2}[item_sub[i]]] for i in range(1, 43)]
fig, ax = plt.subplots(figsize=(14, 4.8))
ax.bar(range(1, 43), dass[items].mean().values, color=colors, edgecolor="white")
ax.set_xticks(range(1, 43, 2))
style_axes(ax, "Item-level mean scores (0-3 after recoding)", "DASS item", "Mean score")
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=PALETTE[i], label=s) for i, s in
                   enumerate(["Depression items", "Anxiety items", "Stress items"])],
          frameon=False, ncol=3, loc="upper center")
save_fig(fig, "exp02_17_dass_items")
print("highest items:", dass[items].mean().nlargest(5).round(2).to_dict())

highest items: {'Q11A': 1.81, 'Q13A': 1.78, 'Q42A': 1.68, 'Q9A': 1.67, 'Q17A': 1.66}


**Takeaway (F17):** item means span only 0.55–1.81 on a 0–3 scale — no item is near ceiling and
the top items (Q11, Q13, Q42) come from different subscales, so Exp 4 must *select* its compact
item set by mutual information rather than by raw means.

## 3 · Text corpora

**Sentiment-MH** (50 843 posts, 7 classes) is the main corpus; **Drug Reviews** (26 832 MH-filtered
reviews) brings rating/sentiment analytics. The Dreaddit overlap (2 819 verbatim posts inside
Sentiment) is handled in Exp 7's training split, not silently merged here.

In [15]:
# F18 class distribution | F19 length distribution per class
g = sent["label"].value_counts().reset_index()
g.columns = ["label", "n"]
fig, ax = plt.subplots(figsize=(9, 5.2))
sns.barplot(data=g, x="label", y="n", color=PALETTE[0], ax=ax)
ax.bar_label(ax.containers[0], fmt="%d", fontsize=9)
style_axes(ax, f"Sentiment corpus class balance (n={len(sent):,})", "Label", "Posts")
ax.tick_params(axis="x", rotation=30)
ax.set_yscale("log")
save_fig(fig, "exp02_18_text_labels")

sent = sent.copy()
sent["wc"] = sent["text_norm"].map(word_count)
order = sent.groupby("label")["wc"].median().sort_values().index
fig, ax = plt.subplots(figsize=(9, 5.2))
sns.boxplot(data=sent, y="label", x="wc", order=order, orient="h", color=PALETTE[1],
            showfliers=False)
ax.set_xlim(0, sent["wc"].quantile(0.99))
style_axes(ax, "Post length by class (99th-percentile view)", "Word count", "Label")
save_fig(fig, "exp02_19_text_length")
print(sent.groupby("label")["wc"].median().astype(int).to_string())

label
Anxiety                 104
Bipolar                 133
Depression              114
Normal                   10
Personality disorder    134
Stress                   87
Suicidal                 92


**Takeaway (F18):** the 7 classes are heavily imbalanced (Normal/Depression ≈ 15 k each vs
Personality disorder < 1 k) — Exp 7 will use class weights and macro-F1.
**Takeaway (F19):** medians range from 10 words (Normal) to 134 (Personality disorder), so length
*is* partially predictive — Exp 7 must publish a length-only baseline to prove its TF-IDF model
is learning vocabulary, not just post length.

In [16]:
# F20 top bigrams per class
grams = top_ngrams(sent["text_norm"], sent["label"], n=2, top_k=6, min_df=10)
items_list = list(grams.items())
n_cols, n_rows = 2, (len(items_list) + 1) // 2
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 3.4 * n_rows))
for ax, (label, pairs) in zip(axes.flat, items_list):
    if not pairs:
        ax.axis("off")
        continue
    toks, vals = zip(*pairs[::-1])
    ax.barh(toks, vals, color=PALETTE[0])
    ax.set_title(label, fontsize=11, fontweight="bold")
    ax.tick_params(labelsize=8)
for ax in axes.flat[len(items_list):]:
    ax.axis("off")
fig.suptitle("Top bigrams per class (document frequency, min_df=10)", fontsize=13, fontweight="bold")
fig.tight_layout()
save_fig(fig, "exp02_20_text_bigrams")
print({k: [g for g, _ in v[:3]] for k, v in grams.items()})

{'Anxiety': ['health anxiety', 'feel like', 'anyone else'], 'Bipolar': ['feel like', 'anyone else', 'right now'], 'Depression': ['feel like', 'right now', 'feels like'], 'Normal': ['feel like', 'even though', "don't know"], 'Personality disorder': ['feel like', 'anyone else', 'feels like'], 'Stress': ['feel like', "don't know", 'right now'], 'Suicidal': ['feel like', 'kill myself', 'want die']}


**Takeaway (F20):** a few classes own distinctive bigrams (*kill myself / want die* → Suicidal,
*health anxiety* → Anxiety), but generic phrases like *feel like* recur across nearly every class
— bigram counts alone won't separate the middle classes; Exp 7 needs full TF-IDF with tuning.

In [17]:
# F21 emotion-lexicon trends per class (curated internal lexicon, rate per 1k tokens)
trend = emotion_trends(sent["text_norm"], sent["label"])
fig, ax = plt.subplots(figsize=(9, 5.4))
sns.heatmap(trend * 1000, annot=True, fmt=".1f", cmap="RdPu", ax=ax,
            cbar_kws={"label": "hits per 1 000 tokens"})
ax.set_title("Emotion-word rates by class", fontweight="bold")
save_fig(fig, "exp02_21_text_emotion")
print((trend * 1000).round(1).to_string())

                       joy  sadness  fear  anger  trust  anticipation
label                                                                
Anxiety                3.0      3.4  31.3    2.5    5.8           5.9
Bipolar                5.6      3.3   4.1    3.8    8.3           6.2
Depression             8.7      9.1   4.4    6.3   10.7           7.9
Normal                11.1      4.3   1.4    3.6    4.6          10.7
Personality disorder   5.8      5.6   6.0    5.0    7.3           7.5
Stress                 4.1      3.6   7.6    4.5   10.9           5.9
Suicidal               7.8      7.9   3.9    7.7   11.4          10.7


**Takeaway (F21):** fear-word rate is 31.3/1k tokens in Anxiety vs 1.4–7.6 elsewhere; sadness
words peak in Depression (9.1) and joy words in Normal (11.1) — an interpretable baseline for
Exp 7's error analysis. The lexicon is small and self-curated (licence-safe), not the NRC lexicon.

In [18]:
# F22 drug ratings by condition | F23 review length + sentiment-vs-rating
topc = drug["condition"].value_counts().head(10)
fig, ax = plt.subplots(figsize=(11, 5.2))
sns.boxplot(data=drug[drug["condition"].isin(topc.index)], x="condition", y="rating",
            order=topc.index, color=PALETTE[3], showfliers=False, ax=ax)
style_axes(ax, "Drug review ratings by condition (top 10 by volume)", "Condition", "Rating (1-10)")
ax.tick_params(axis="x", rotation=35)
save_fig(fig, "exp02_22_drug_ratings")

drug = drug.copy()
drug["wc"] = drug["text_norm"].map(word_count)
median_len = drug[drug["condition"].isin(topc.index)].groupby("condition")["wc"].median().reindex(topc.index)
drug["vader"] = vader_compound(drug["text_norm"]).values
by_rating = drug.groupby("rating")["vader"].mean()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh(median_len.index[::-1], median_len.values[::-1], color=PALETTE[1])
style_axes(axes[0], "Median review length by condition", "Words", "Condition")
axes[1].plot(by_rating.index, by_rating.values, marker="o", linewidth=2, color=PALETTE[4])
style_axes(axes[1], "Mean VADER sentiment vs star rating", "Rating", "VADER compound")
axes[1].set_xticks(range(1, 11))
save_fig(fig, "exp02_23_drug_sentiment")
from scipy.stats import spearmanr
rho = spearmanr(drug["rating"], drug["vader"]).statistic
print(f"Spearman rating ~ VADER: {rho:.3f}")
print("median words:", median_len.astype(int).to_dict())

Spearman rating ~ VADER: 0.274
median words: {'Depression': 93, 'Anxiety': 86, 'Insomnia': 71, 'Bipolar Disorde': 81, 'ADHD': 100, 'Anxiety and Stress': 103, 'Major Depressive Disorde': 100, 'Panic Disorde': 96, 'Generalized Anxiety Disorde': 108, 'Social Anxiety Disorde': 106}


**Takeaway (F22):** ratings are bimodal (9–10 k vs 1–2 stars, thin middle) — a plain regression
on rating would be poorly calibrated; Exp 7 treats these analytics as descriptive only.
**Takeaway (F23):** VADER correlates only *modestly* with star ratings (Spearman ρ = 0.27) —
expected, since reviews discuss side effects rather than simple good/bad — so sentiment is a
secondary cue in the app, with the text model as primary.

## 4 · Face images (FER-2013 manifest, N = 35 887)

The manifest lists file paths (never copied into the repo) plus the 7 emotion labels and the
official train/test split. The pixel statistics below read a bounded sample for speed.

In [19]:
# F24 class counts + per-image brightness distribution
from PIL import Image

counts_fer = fer["emotion"].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
sns.barplot(x=counts_fer.index, y=counts_fer.values, color=PALETTE[0], ax=axes[0])
axes[0].bar_label(axes[0].containers[0], fmt="%d", fontsize=9)
style_axes(axes[0], "FER-2013 class counts", "Emotion", "Images")
axes[0].tick_params(axis="x", rotation=30)

sample = fer.sample(n=min(600, len(fer)), random_state=GLOBAL_SEED)
brightness = [
    np.asarray(Image.open(repo_path(p)).convert("L"), dtype=float).mean()
    for p in sample["path"]
]
axes[1].hist(brightness, bins=40, color=PALETTE[1], edgecolor="white")
style_axes(axes[1], f"Per-image mean brightness (sample n={len(brightness)})", "Mean pixel (0-255)", "Images")
save_fig(fig, "exp02_24_fer_counts")
print(counts_fer.to_string())
print("brightness mean/std:", round(float(np.mean(brightness)), 1), "/", round(float(np.std(brightness)), 1))

emotion
happy       8989
neutral     6198
sad         6077
fear        5121
angry       4953
surprise    4002
disgust      547
brightness mean/std: 128.7 / 32.8


**Takeaway (F24):** a 16:1 happy/disgust ratio (8 989 vs 547) plus mean brightness 128.7 ± 32.8
confirm class weighting is mandatory and exposure varies — reasons Exp 3 reports macro-F1 only
(human accuracy on FER-2013 is itself ~65%) and the app labels the output a mood *cue*.

In [20]:
# F25 sample grid: 6 faces per emotion
emos = sorted(fer["emotion"].unique())
fig, axes = plt.subplots(len(emos), 6, figsize=(12, 2.1 * len(emos)))
for i, e in enumerate(emos):
    paths = list(fer.loc[fer["emotion"] == e, "path"].head(6))
    for j in range(6):
        ax = axes[i, j]
        ax.axis("off")
        if j < len(paths):
            ax.imshow(Image.open(repo_path(paths[j])).convert("L"), cmap="gray")
        if j == 0:
            ax.set_ylabel(e, fontsize=10, rotation=0, ha="right", va="center", labelpad=8)
fig.suptitle("FER-2013 samples (48x48 grayscale)", fontsize=13, fontweight="bold", y=1.0)
fig.tight_layout()
save_fig(fig, "exp02_25_fer_grid")
print("grid written for", len(emos), "classes")

grid written for 7 classes


**Takeaway (F25):** at 48×48 grayscale the images carry a mood *cue* only — resolution, lighting
and demographic gaps are why Exp 3 and the app label the face signal **non-diagnostic**.

## 5 · Interactive views for the Population Insights page

Two Plotly figures are saved as self-contained HTML (`reports/figures/exp02_insights_*.html`).
Filters: click legend entries to include/exclude a gender; facets split the population — the same
filter pattern the Streamlit page re-creates from the parquet.

In [21]:
import plotly.express as px

agg = (add_age_band(risk).groupby(["population", "gender", "age_band"], observed=True)
       .agg(rate=("mh_risk", "mean"), n=("mh_risk", "size")).reset_index())
fig1 = px.bar(
    agg, x="age_band", y="rate", color="gender", facet_col="population", barmode="group",
    hover_data=["n"], category_orders={"age_band": age_order, "gender": sorted(risk["gender"].unique())},
    title="MindSense -- risk rate by age band (filter via legend)",
    labels={"rate": "Risk rate", "age_band": "Age band"},
)
fig1.update_yaxes(tickformat=".0%")
p1 = save_plotly_html(fig1, "exp02_insights_risk")

fig2 = px.histogram(
    add_age_band(dass), x="depression_band", color="gender", facet_col="age_band",
    category_orders={"depression_band": BAND_ORDER},
    barmode="stack", title="DASS depression bands by age band (filter via legend)",
    labels={"depression_band": "Depression band", "count": "Respondents"},
)
p2 = save_plotly_html(fig2, "exp02_insights_dass")
print("saved:", p1, "|", p2)

saved: /Users/pranavghodke/Desktop/Honors_Mini_Project/reports/figures/exp02_insights_risk.html | /Users/pranavghodke/Desktop/Honors_Mini_Project/reports/figures/exp02_insights_dass.html


**Takeaway:** both HTML views open in any browser with hover read-outs and legend/facet filters —
they are the offline fallback for the app's Population Insights tab.

## 6 · Headline metrics for the Lab Results page

In [22]:
mi = mutual_information_ranking(
    risk, "mh_risk",
    ["age", "gender", "population", "sleep_hours", "stress_pressure_score", "satisfaction_score",
     "work_study_hours", "financial_stress", "family_history", "support_available",
     "diet_quality", "suicidal_thoughts"],
    categorical=["gender", "population", "family_history", "diet_quality", "suicidal_thoughts"],
    seed=GLOBAL_SEED,
)
metrics = {
    "generated": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "seed": GLOBAL_SEED,
    "data_source": dict(counts),
    "figures": sorted(p.name for p in repo_path("reports", "figures").glob("exp02_*.png")),
    "n_interactive_html": len(list(repo_path("reports", "figures").glob("exp02_insights_*.html"))),
    "target": {
        "mh_risk_rate": round(float(risk["mh_risk"].mean()), 4),
        "rate_by_population": risk.groupby("population")["mh_risk"].mean().round(4).to_dict(),
        "n": int(len(risk)),
    },
    "mi_top5": {k: round(float(v), 4) for k, v in mi.head(5).items()},
    "dass_depression_band_share": {
        k: round(float(v), 4) for k, v in dass["depression_band"].value_counts(normalize=True).items()
    },
    "dass_n_after_validity_screen": int(len(dass)),
    "text_class_balance": {k: int(v) for k, v in sent["label"].value_counts().items()},
    "drug": {
        "n": int(len(drug)),
        "rating_mean": round(float(drug["rating"].mean()), 2),
        "vader_rating_spearman": round(float(spearmanr(drug["rating"], drug["vader"]).statistic), 4),
    },
    "fer": {"n": int(len(fer)), "classes": {k: int(v) for k, v in fer["emotion"].value_counts().items()}},
}
out = save_json(metrics, repo_path("reports", "metrics", "exp02_eda.json"))
print("wrote", out)
print(json.dumps({k: metrics[k] for k in ("target", "mi_top5", "dass_n_after_validity_screen")}, indent=1))

wrote /Users/pranavghodke/Desktop/Honors_Mini_Project/reports/metrics/exp02_eda.json
{
 "target": {
  "mh_risk_rate": 0.5821,
  "rate_by_population": {
   "professional": 0.5052,
   "student": 0.5855
  },
  "n": 29152
 },
 "mi_top5": {
  "suicidal_thoughts": 0.1407,
  "stress_pressure_score": 0.1258,
  "financial_stress": 0.0664,
  "age": 0.0315,
  "work_study_hours": 0.0255
 },
 "dass_n_after_validity_screen": 38337
}


## Observations & Limitations

- **Proxy everywhere:** `mh_risk` blends a student self-report label with a professional
  treatment indicator; DASS shares come from a self-selected online survey. None of these numbers
  estimate clinical prevalence (Rule 5).
- **Structural missingness is not noise:** sleep/finance/diet are student-only and
  workplace-support is professional-only in the harmonized table; imputation across populations
  would invent data (DECISIONS #8).
- **Imbalance:** the tabular target is ~58/42, the text classes are ~15 k vs <1 k, FER has a
  13:1 happy/disgust ratio — every downstream metric choice follows from this.
- **Cross-corpus contamination:** 2 819 verbatim Dreaddit posts live inside the Sentiment corpus;
  Exp 7 must exclude them from training and re-measure.
- **Subgroup smallness:** `gender = other` (~21 rows) and the smallest DASS countries are shown
  for completeness but are not stable enough for fairness claims — flagged for Exp 8.
- **Emotion lexicon** is a small internal word list (licence-safe), not a validated clinical
  instrument; VADER is a general sentiment tool, not a mental-health model.
- All figures and both interactive views were produced by executing this notebook end to end on
  the processed tables (seed 42).

## Conclusion

EDA confirms the design decisions the rest of the project rests on: a workable (imbalanced)
tabular target with population-specific feature blocks, a DASS distribution that motivates
ordinal metrics and item reduction, text corpora with strong lexical class separation and known
cross-corpus overlap, and an image set whose imbalance and resolution demand a "mood cue, not a
diagnosis" framing. Artefacts: 25 figures, 2 interactive HTML views and
`reports/metrics/exp02_eda.json` for the Lab Results page.